# Load sc distances

In [9]:
import glob
import os
import re
import numpy as np
import pandas as pd
from topsy.analysis.compare_distances import load_sc_dis_per_locus as load_sc_dis_per_locus_tmp
from topsy.analysis.compare_distances import load_sc_struct_features as load_sc_struct_features_tmp
# from topsy.analysis.compare_distances import scale_sc_distances as scale_sc_distances_tmp


# counts_subdir = 'astro.nreads1e8.sc_alpha-3'
# counts_subdir = 'astro.nreads1e9.logistic_k-14.71_m0.4515'

# IMPUTED, NOT NORMALIZED
counts_subdir = 'astro.nreads1e9.logistic_k-5.16_m2.69_v1.23e-05'  # obj=-0.88644655 → PEARSON.infer_nu.kmax-4_buffer1e-5_d0min50p, seed=0  (FACTR) - IMPUTED, NOT normalized
# counts_subdir = 'astro.nreads1e9.logistic_k-9.12_m1.72_v1e-05'  # obj=0.18979348 → RMSE.infer_nu.kmax-4_buffer1e-5_d0min50p, seed=0  (FACTR) - IMPUTED, NOT normalized

sc_dis_file = f'/net/gs/vol1/home/gesine/noble_lab/projects/2015_gesine_diploid/results/liu2025/unambig/{counts_subdir}/struct_true.distances.per_locus.tsv'
sc_features_file = f'/net/gs/vol1/home/gesine/noble_lab/projects/2015_gesine_diploid/results/liu2025/unambig/{counts_subdir}/struct_true.features.tsv'

struct_infer_file = f'/net/gs/vol1/home/gesine/noble_lab/projects/2015_gesine_diploid/results/liu2025/unambig/{counts_subdir}/infer_ua_filter0perc.singleres/struct_inferred.000.coords'
# dataset_info_file = f'/net/gs/vol1/home/gesine/noble_lab/projects/2015_gesine_diploid/data/liu2025/unambig/{counts_subdir}/dataset_info.txt'
dataset_dir = f'/net/gs/vol1/home/gesine/noble_lab/projects/2015_gesine_diploid/results/liu2025/unambig/{counts_subdir}/'

struct_infer_glob = f'/net/gs/vol1/home/gesine/noble_lab/projects/2015_gesine_diploid/results/liu2025/unambig/{counts_subdir}/infer_*/struct_inferred.*.coords'
# struct_infer_files = sorted(glob.glob(struct_infer_glob))

# sc_dis = load_sc_dis_per_locus_tmp(sc_dis_file, scale=False, verbose=True).drop('same_molecule', axis=1)
# sc_dis = scale_sc_distances_tmp(sc_dis, dataset_dir=dataset_dir, copy=False, verbose=True)

In [2]:
sc_features = load_sc_struct_features_tmp(sc_features_file, verbose=True)

sc_dis = load_sc_dis_per_locus_tmp(sc_dis_file, verbose=True).drop('same_molecule', axis=1)
# sc_dis = scale_sc_distances_tmp(sc_dis, dataset_dir=dataset_dir, copy=False, verbose=True)

Loading sc structural features... DONE
Loading sc distances... DONE


# Code

In [3]:
import os
import re
import ast
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist
from topsy.analysis.utils import get_nghbr_dis_var, _get_mse


def make_diploid_lengths_df(lengths_df):
    if isinstance(lengths_df, str):
        lengths_df = pd.read_csv(lengths_df, sep="\t")
        lengths_df.columns = [c.replace('#', '') for c in lengths_df.columns]

    tmp = lengths_df.copy()
    tmp['idx'] = tmp.idx_genome
    tmp['hmlg'] = 1
    diploid_lengths_df = tmp.copy()
    diploid_lengths_df['idx'] += len(lengths_df)
    diploid_lengths_df['hmlg'] += 1
    diploid_lengths_df = pd.concat([
        tmp, diploid_lengths_df]).reset_index(drop=True)
    diploid_lengths_df['mol'] = diploid_lengths_df.chrom + '.' + \
        diploid_lengths_df.hmlg.astype(str)

    return diploid_lengths_df


def make_matrix_df(lengths_df, ploidy=2, distance_dict=None, matrix_dict=None):
    if isinstance(lengths_df, str):
        lengths_df = pd.read_csv(lengths_df, sep="\t")
        lengths_df.columns = [c.replace('#', '') for c in lengths_df.columns]

    lengths = lengths_df.groupby('chrom').size().sort_values(
        ascending=False).values
    nbeads = lengths.sum() * ploidy

    if ploidy == 2:
        lengths_df = make_diploid_lengths_df(lengths_df)

    rows, cols = np.triu_indices(nbeads, 1)
    df = pd.DataFrame()
    df['i.idx'] = lengths_df.idx.values[rows]
    df['j.idx'] = lengths_df.idx.values[cols]
    df['i.idx_chrom'] = lengths_df.idx_chrom.values[rows]
    df['j.idx_chrom'] = lengths_df.idx_chrom.values[cols]
    df['i.chrom'] = lengths_df.chrom.values[rows]
    df['j.chrom'] = lengths_df.chrom.values[cols]
    if ploidy == 2:
        df['i.idx_ambig'] = lengths_df.idx_genome.values[rows]
        df['j.idx_ambig'] = lengths_df.idx_genome.values[cols]
        df['i.hmlg'] = lengths_df.hmlg.values[rows]
        df['j.hmlg'] = lengths_df.hmlg.values[cols]

    # diffM, sameC-sameH, sameC-diffH, diffC-sameH, diffC-diffH
    sameC = df['i.chrom'] == df['j.chrom']
    if ploidy == 1:
        df['mask.sameM'] = sameC
        df['mask.diffM'] = ~sameC
    else:
        sameH = df['i.hmlg'] == df['j.hmlg']
        df['mask.diffM'] = (~sameC) | (~sameH)
        df['mask.sameC-sameH'] = ~df['mask.diffM']  # aka sameM
        df['mask.sameC-diffH'] = sameC & (~sameH)  # included in diffM
        df['mask.diffC-sameH'] = (~sameC) & sameH  # included in diffM
        df['mask.diffC-diffH'] = (~sameC) & (~sameH)  # included in diffM

    df['genomic_dis'] = None
    df.loc[df['mask.sameC-sameH'], 'genomic_dis'] = (
        df['i.idx'] - df['j.idx']).abs()
    df['mask.nghbr'] = df['mask.sameC-sameH'] & (df.genomic_dis == 1)

    df.index = list(map(tuple, np.stack(
        [df['i.idx'], df['j.idx']], axis=1).tolist()))

    if distance_dict is not None:
        for name, dist in distance_dict.items():
            df[name] = dist
    if matrix_dict is not None:
        for name, matrix in matrix_dict.items():
            # df[name] = matrix[triu]
            df[name] = matrix[(df['i.idx'].values, df['j.idx'].values)]

    return df


def establish_hmlg_order(matrix_df, res_df, swap_df):
    n = matrix_df.loc[
        matrix_df['i.hmlg'] == 2, 'i.idx'].values[0] - matrix_df.loc[
        matrix_df['i.hmlg'] == 2, 'i.idx_ambig'].values[0]

    # Error score that determines whether homologs should be swapped
    if len(res_df.columns) == 1:
        err_score = res_df.columns.values[0]
    else:
        err_score = 'disterror.vs_all'

    mask = matrix_df['mask.sameC-sameH']
    per_chrom_unswapped = res_df[mask].groupby(
        matrix_df.loc[mask, 'i.chrom']).mean()
    per_chrom_swapped = swap_df[mask].groupby(
        matrix_df.loc[mask, 'i.chrom']).mean()
    swap_chrom = per_chrom_unswapped > per_chrom_swapped
    chrom_to_swap = swap_chrom[swap_chrom[err_score]].index

    if len(chrom_to_swap):
        print(f"Swapping homolog labels for: {', '.join(chrom_to_swap)}",
              flush=True)
    elif np.allclose(per_chrom_unswapped.values, per_chrom_swapped.values):
        print("Results are near-identical regardless of homolog labels",
              flush=True)
    else:
        print("Homolog labels were not swapped for any chrom", flush=True)

    for chrom in chrom_to_swap:
        for locus in ('i', 'j'):
            chrom_mask = mask & (matrix_df[f"{locus}.chrom"] == chrom)
            matrix_df.loc[chrom_mask, 'swapped'] += 1
            # Swap the homolog labels
            matrix_df.loc[chrom_mask, f"{locus}.hmlg"] = np.invert((
                matrix_df.loc[chrom_mask, f"{locus}.hmlg"] - 1).astype(
                bool)).astype(int) + 1
            # Update locus idx labels to match
            matrix_df.loc[chrom_mask, f"{locus}.idx"] = matrix_df.loc[
                chrom_mask, f"{locus}.idx_ambig"] + n * (
                    matrix_df.loc[chrom_mask, f"{locus}.hmlg"] - 1)
        # Incorporate similarity scores obtained from chrom-swapped data
        res_df[mask & (matrix_df['i.chrom'] == chrom)] = swap_df[mask & (
            matrix_df['i.chrom'] == chrom)]
    if len(chrom_to_swap):
        # Update DataFrame index to match
        matrix_df.index = list(map(tuple, np.stack(
            [matrix_df['i.idx'], matrix_df['j.idx']], axis=1).tolist()))

    for col in res_df.columns:
        matrix_df[col] = res_df[col]

    return matrix_df


def distances_infer_vs_true(matrix_df, sc_dis, rescale=True,
                            guess_hmlg_order=True, verbose=True):

    if verbose:
        if guess_hmlg_order:
            print("Comparing intra-molecular distance bins and matching up"
                  " homologs", flush=True)
        else:
            print("Comparing all distance bins", flush=True)

    matrix_df = matrix_df.loc[sc_dis.index]  # Filter & order to match sc_dis

    if rescale:  # Rescale using distances between neighbors as reference
        mask_nghbr = matrix_df['mask.sameC-sameH'] & (
            matrix_df.genomic_dis == 1)
        Y = sc_dis.loc[mask_nghbr, 'dis']
        if guess_hmlg_order:
            X = matrix_df.loc[mask_nghbr, ['dis_infer', 'dis_infer.swap']]
        else:
            X = matrix_df.loc[mask_nghbr, 'dis_infer']

        print(X.describe().to_string() + '\n')
        print(X.quantile(.95), X.quantile(.96), X.quantile(.97), X.quantile(.98))
        # print(matrix_df.loc[mask_nghbr & (X > X.quantile(.95))].drop([
        #     'i.idx', 'j.idx', 'i.idx_ambig', 'j.idx_ambig',
        #     'genomic_dis'] + [c for c in matrix_df.columns if c.startswith(
        #         'mask.')], axis=1))
        tmp = matrix_df.loc[mask_nghbr & (X > X.quantile(.98)), [
            'i.idx_chrom', 'j.idx_chrom', 'i.chrom', 'i.hmlg', 'dis_infer',
            'i.idx_ambig', 'j.idx_ambig']]
        print(tmp.drop(['i.idx_ambig', 'j.idx_ambig'], axis=1))
        print()
        foo = tmp[tmp['j.idx_ambig'].isin(tmp['i.idx_ambig'].values)]
        print(foo)
        print('\n')
        
        scale_factor = (X * Y).apply(np.mean).sum() / X.pow(2).sum()
        matrix_df['dis_infer'] *= scale_factor
        if guess_hmlg_order:
            matrix_df['dis_infer.swap'] *= scale_factor
        if verbose:
            print(f"Rescale inferred struct by {scale_factor:.3g}", flush=True)
            print('pre-rescaling: dis_infer', X.mean())
            print(matrix_df.loc[mask_nghbr, ['dis_infer']].mean().to_string())
            print(sc_dis.loc[mask_nghbr, ['dis_mean', 'dis_med']].mean(axis=0).to_string())
            Y = sc_dis.loc[mask_nghbr, 'dis_mean']
            print(f'if rescale by dis_mean: {(X * Y).sum() / X.pow(2).sum():.3g}\n')
        raise ValueError('stop here')

    for col in ['disterror.vs_all', 'disterror.vs_mean', 'disterror.vs_med',
                'dis_true_mean', 'dis_true_med']:
        if col not in matrix_df.columns:
            matrix_df[col] = np.nan
    if 'swapped' not in matrix_df.columns:
        matrix_df['swapped'] = 0

    if guess_hmlg_order:
        res_df = matrix_df[[
            'disterror.vs_all', 'disterror.vs_mean', 'disterror.vs_med']].copy()
        swap_df = res_df.copy()
    else:
        res_df = matrix_df
        swap_df = None

    mask = matrix_df.dis_true_mean.isnull()
    if guess_hmlg_order:  # Only using intra-molecular to guess homolog order
        mask = mask & matrix_df['mask.sameC-sameH']

    matrix_df.loc[mask, 'dis_true_mean'] = sc_dis.loc[mask, 'dis_mean']
    matrix_df.loc[mask, 'dis_true_med'] = sc_dis.loc[mask, 'dis_med']
    dis_true = sc_dis.loc[mask, 'dis']

    dis_infer = matrix_df.loc[mask, 'dis_infer']
    res_df.loc[mask, 'disterror.vs_mean'] = (
        dis_infer - matrix_df.loc[mask, 'dis_true_mean']).pow(2)
    res_df.loc[mask, 'disterror.vs_med'] = (
        dis_infer - matrix_df.loc[mask, 'dis_true_med']).pow(2)
    res_df.loc[mask, 'disterror.vs_all'] = (
        dis_infer - dis_true).pow(2).apply(np.mean)
    if guess_hmlg_order:
        dis_infer = matrix_df.loc[mask, 'dis_infer.swap']  # Homologs swapped
        swap_df.loc[mask, 'disterror.vs_mean'] = (
            dis_infer - matrix_df.loc[mask, 'dis_true_mean']).pow(2)
        swap_df.loc[mask, 'disterror.vs_med'] = (
            dis_infer - matrix_df.loc[mask, 'dis_true_med']).pow(2)
        swap_df.loc[mask, 'disterror.vs_all'] = (dis_infer - dis_true).pow(
            2).apply(np.mean)

    if guess_hmlg_order:
        matrix_df = establish_hmlg_order(
            matrix_df, res_df=res_df, swap_df=swap_df)
        if rescale:
            # Re-calculate all error scores using updated scale factor
            matrix_df['dis_true_mean'] = None
        return distances_infer_vs_true(
            matrix_df, sc_dis=sc_dis, rescale=rescale, guess_hmlg_order=False,
            verbose=verbose)

    return matrix_df


def get_other_struct_features(matrix_df, dis_df, ploidy=2):
    if isinstance(dis_df, str):  # Get distance data from column of matrix_df
        dis_df = matrix_df[[dis_df]]
    else:
        matrix_df = matrix_df.loc[dis_df.index]  # Sort/filter to match dis_df
    if isinstance(dis_df, pd.Series):
        dis_df = dis_df.to_frame()

    if matrix_df['mask.diffM'].sum():  # Only using intra-molecular distances
        sameM = ~matrix_df['mask.diffM'].values
        matrix_df = matrix_df[sameM]
        dis_df = dis_df[sameM]

    results = {}

    # Characterize distances between neighboring beads
    mask_nghbr = (
        matrix_df['i.idx_ambig'] - matrix_df['j.idx_ambig']).abs() == 1
    results['nghbrdis_mean'] = dis_df[mask_nghbr].mean(axis=0).values
    results['nghbrdis_var'] = dis_df[mask_nghbr].apply(
        get_nghbr_dis_var, axis=0).values

    # How different are the homologs? MSE of Distance error between homologs
    if ploidy == 2:
        n = matrix_df.loc[
            matrix_df['i.hmlg'] == 2, 'i.idx'].values[0] - matrix_df.loc[
            matrix_df['i.hmlg'] == 2, 'i.idx_ambig'].values[0]
        idx_hmlg1 = matrix_df[matrix_df['i.hmlg'] == 1].index
        idx_hmlg2 = list(map(tuple, (
            np.array(idx_hmlg1.values.tolist()) + n).tolist()))

        sq_diff = np.square(
            dis_df.loc[idx_hmlg1].values - dis_df.loc[idx_hmlg2].values)
        results['btwn-hmlgs_disterror'] = np.sqrt(np.nanmean(
            sq_diff[:, ~np.all(np.isnan(sq_diff), axis=0)], axis=0))

    if len(dis_df.columns) == 1:
        results = {k: v[0] for k, v in results.items()}

    return results


def compare_infer_vs_true(struct_infer_file, sc_dis=None, sc_features=None,
                          rescale=True, hmlg_order_known=None, redo=False,
                          verbose=True):

    # Get output files
    outdir = os.path.dirname(struct_infer_file)
    struct_desc = re.sub(
        r'\.coords(\.gz)*$', '', os.path.basename(struct_infer_file)).replace(
        'struct_inferred.', 'infer').replace('struct_init.', 'init')
    if rescale:
        struct_desc += '.rescaled'
    outfile_perbin = os.path.join(
        outdir, f"disterror_per_bin.true_vs_{struct_desc}")
    outfile_features = os.path.join(
        outdir, f"compare_features.true_vs_{struct_desc}")
    outfile_scores = os.path.join(outdir, f"error.true_vs_{struct_desc}")
    print(re.sub(
        '^.*2015_gesine_diploid/(results/){0,1}', '',
        outfile_perbin).replace(os.environ['HOME'], '~'), flush=True)

    # Get inference type
    infer_dir = get_inference_dir(struct_infer_file)
    inference_type = re.sub(r'^infer_', '', os.path.basename(infer_dir)).replace(
        'alpha', 'α').replace('_filter0perc', '').replace('perc', '%')

    if (not redo) and os.path.isfile(outfile_perbin) and os.path.isfile(
            outfile_features) and os.path.isfile(outfile_scores):
        matrix_df = pd.read_csv(
            outfile_perbin, sep='\t', index_col=0,
            converters={0: ast.literal_eval})
        features_df = pd.read_csv(outfile_features, sep='\t', index_col=0)
        error = pd.read_csv(
            outfile_scores, sep='\t', header=None, index_col=0).squeeze(
            "columns")
        return matrix_df, features_df, error, inference_type

    # Load data
    matrix_df, dataset_dir, infer_dir, hmlg_order_known = load_inferred_dis(
        struct_infer_file, hmlg_order_known=hmlg_order_known, verbose=verbose)
    if sc_dis is None:
        sc_dis = sc_dis = os.path.join(
            dataset_dir, "struct_true.distances.per_locus.tsv")
    if isinstance(sc_dis, str):
        sc_dis = load_sc_dis_per_locus(sc_dis, verbose=verbose).drop(
            'same_molecule', axis=1)
    if sc_features is None:
        sc_features = os.path.join(dataset_dir, "struct_true.features.tsv")
    if isinstance(sc_features, str):
        sc_features = load_sc_struct_features(sc_features, verbose=verbose)

    # Get distance error between inferred structure & true structures
    if (not redo) and os.path.isfile(outfile_perbin):
        matrix_df = pd.read_csv(
            outfile_perbin, sep='\t', index_col=0,
            converters={0: ast.literal_eval})
    else:
        matrix_df = distances_infer_vs_true(
            matrix_df=matrix_df, sc_dis=sc_dis, rescale=rescale,
            guess_hmlg_order=not hmlg_order_known, verbose=verbose)
        matrix_df.to_csv(outfile_perbin, sep='\t')
        redo = True  # Redo scores computed below even if file(s) exist
    disterror_res = get_disterror_from_sq_dis(matrix_df, verbose=verbose - 1)

    # Compare other structural features TODO
    if (not redo) and os.path.isfile(outfile_features):
        features_df = pd.read_csv(outfile_features, sep='\t', index_col=0)
    else:
        features_df = pd.DataFrame(
            index=np.array(
                [[f"{x}_mse", f"{x}_err"] for x in sc_features.index]).ravel(),
            columns=[x.replace('sc_', 'vs_') for x in sc_features.columns])
        inferred_features = get_other_struct_features(
            matrix_df, dis_df='dis_infer')
        for feature_name, val_infer in inferred_features.items():
            vals_sc = sc_features.loc[feature_name]
            for sc_agg in sc_features.columns:
                col = sc_agg.replace('sc_', 'vs_')
                features_df.loc[f"{feature_name}_mse", col] = _get_mse(
                    X=val_infer, Y=vals_sc[sc_agg])
                features_df.loc[f"{feature_name}_err", col] = _get_mse(
                    X=val_infer, Y=vals_sc[sc_agg], square=False)
        features_df.to_csv(outfile_features, sep='\t')
        redo = True  # Redo scores computed below even if file exists

    if (not redo) and os.path.isfile(outfile_scores):
        error = pd.read_csv(
            outfile_scores, sep='\t', header=None, index_col=0).squeeze(
            "columns")
    else:
        error = pd.concat([
            disterror_res['disterror.vs_all'], features_df['vs_all']])
        error.to_csv(outfile_scores, sep='\t', header=False)

    return matrix_df, features_df, error, inference_type


def get_disterror_from_sq_dis(matrix_df, verbose=True):
    disterror_types = [c.replace(
        'mask.', '') for c in matrix_df.columns if c.startswith('mask.')]
    disterror_res = pd.DataFrame(
        columns=[c for c in matrix_df.columns if c.startswith('disterror.')],
        index=sorted(['all'] + disterror_types))
    for col in disterror_res.columns:
        disterror_res.loc['all', col] = matrix_df[col].mean()
        for err_type in disterror_types:
            disterror_res.loc[err_type, col] = np.sqrt(matrix_df.loc[
                matrix_df[f"mask.{err_type}"], col].mean())
    if verbose:
        print(disterror_res.to_string(), flush=True)
    return disterror_res


def get_dataset_dir(struct_infer_files):
    if isinstance(struct_infer_files, str):
        struct_infer_files = [struct_infer_files]

    dataset_dir = os.path.commonpath([
        os.path.dirname(x) for x in struct_infer_files])
    i = 0
    while i < 10 and not os.path.isfile(os.path.join(
            dataset_dir, 'counts.bed')):
        dataset_dir = os.path.dirname(dataset_dir)
        i += 1
    if not os.path.isfile(os.path.join(dataset_dir, 'counts.bed')):
        raise ValueError("Couldn't find directory with dataset files...")

    return dataset_dir


def get_inference_dir(struct_infer_files):
    if isinstance(struct_infer_files, str):
        struct_infer_files = [struct_infer_files]

    infer_dir = os.path.commonpath([
        os.path.dirname(x) for x in struct_infer_files])
    i = 0
    while i < 10 and not os.path.isfile(os.path.join(
            infer_dir, 'config.pastis')):
        infer_dir = os.path.dirname(infer_dir)
        i += 1
    if not os.path.isfile(os.path.join(infer_dir, 'config.pastis')):
        raise ValueError("Couldn't find directory with 'config.pastis'...")

    return infer_dir


def load_inferred_dis(struct_infer_file, hmlg_order_known=None, verbose=True):
    dataset_dir = get_dataset_dir(struct_infer_file)
    infer_dir = get_inference_dir(struct_infer_file)

    lengths_df = pd.read_csv(os.path.join(dataset_dir, 'counts.bed'), sep="\t")
    lengths_df.columns = [c.replace('#', '') for c in lengths_df.columns]
    lengths = lengths_df.groupby('chrom').size().sort_values(
        ascending=False).values
    n = lengths.sum()

    if hmlg_order_known is None:
        config_file = os.path.join(infer_dir, "config.pastis")
        counts_files = [os.path.basename(x).lower() for x in pd.read_csv(
            config_file, sep='\t', header=None, index_col=0).squeeze(
            "columns")["counts"].split(' ')]
        hmlg_order_known = any([
            ('counts_ua' in x or 'ua_counts' in x) for x in counts_files])
        if verbose:
            print("Assuming homolog ordering of inferred structure has "
                  f"{'' if hmlg_order_known else 'NOT '}been pre-established",
                  flush=True)

    struct_infer = np.loadtxt(struct_infer_file)

    distance_dict = {'dis_infer': pdist(struct_infer)}
    if not hmlg_order_known:
        struct_infer_swap = np.concatenate([struct_infer[n:], struct_infer[:n]])
        distance_dict['dis_infer.swap'] = pdist(struct_infer_swap)

    matrix_df = make_matrix_df(lengths_df, distance_dict=distance_dict)
    matrix_df = matrix_df[~matrix_df.dis_infer.isnull()]

    return matrix_df, dataset_dir, infer_dir, hmlg_order_known


def load_sc_dis_per_locus(sc_dis_file, verbose=True):
    if os.path.isfile(f'{sc_dis_file}.gz') and not os.path.isfile(sc_dis_file):
        sc_dis_file = f'{sc_dis_file}.gz'
    if verbose:
        print("Loading sc distances... ", end="", flush=True)
    sc_dis = pd.read_csv(
        sc_dis_file, sep='\t', header=None, converters={
            0: ast.literal_eval, 1: int, 2: float, 3: float,
            4: ast.literal_eval},
        names=('idx', 'same_molecule', 'dis_mean', 'dis_med', 'dis')
    ).set_index('idx')
    sc_dis['dis'] = sc_dis.dis.apply(np.array)
    if verbose:
        print("DONE", flush=True)
    return sc_dis


def load_sc_struct_features(sc_features_file, verbose=True):
    if os.path.isfile(f'{sc_features_file}.gz') and not os.path.isfile(
            sc_features_file):
        sc_features_file = f'{sc_features_file}.gz'
    if verbose:
        print("Loading sc structural features... ", end="", flush=True)
    sc_features = pd.read_csv(
        sc_features_file, sep='\t', index_col=0, converters={
            0: str, 1: float, 2: float, 3: ast.literal_eval})
    sc_features['sc_all'] = sc_features['sc_all'].apply(np.array)
    if verbose:
        print("DONE", flush=True)
    return sc_features


def get_sc_disterror_all(struct_infer_files, sc_dis=None, sc_features=None,
                         rescale=True, hmlg_order_known=None, redo=False,
                         verbose=True):
    if isinstance(struct_infer_files, str):
        struct_infer_files = [struct_infer_files]
    dataset_dir = get_dataset_dir(struct_infer_files)

    if sc_dis is None:
        sc_dis = os.path.join(
            dataset_dir, "struct_true.distances.per_locus.tsv")
    if isinstance(sc_dis, str):
        sc_dis = load_sc_dis_per_locus(sc_dis, verbose=verbose).drop(
            'same_molecule', axis=1)
    if sc_features is None:
        sc_features = os.path.join(dataset_dir, "struct_true.features.tsv")
    if isinstance(sc_features, str):
        sc_features = load_sc_struct_features(sc_features, verbose=verbose)

    disterror_res_mean = []
    for struct_infer_file in struct_infer_files:
        matrix_df, features_df, error, inference_type = compare_infer_vs_true(
            struct_infer_file, sc_dis=sc_dis, sc_features=sc_features,
            rescale=rescale, hmlg_order_known=hmlg_order_known, redo=redo,
            verbose=verbose)
        disterror_res = get_disterror_from_sq_dis(matrix_df, verbose=False)
        disterror_res['inference'] = inference_type
        disterror_res_mean.append(disterror_res)
        if verbose and struct_infer_file != struct_infer_files[-1]:
            print(flush=True)
    if len(struct_infer_files) > 1:
        disterror_res_mean = pd.concat(disterror_res_mean).reset_index().rename(
            {'index': 'err'}, axis=1).groupby(['inference', 'err']).mean()
        if verbose:
            print('\nMean results:')
            print(disterror_res_mean, flush=True)


def main():
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--struct_infer", type=str, nargs='+',
                        help="Inferred structure file(s)")
    parser.add_argument("--dis_true", type=str,
                        help="Single-cell 'true' distances (per locus)")
    parser.add_argument("--features_true", type=str,
                        help="Structural features of single-cell 'true'"
                             " structures")
    parser.add_argument('--rescale', dest='rescale',
                        default=True, action='store_true')
    parser.add_argument('--dont-rescale', dest='rescale',
                        default=True, action='store_false')
    parser.add_argument('--ordered', dest='hmlg_order_known',
                        default=None, action='store_true')
    parser.add_argument('--not-ordered', dest='hmlg_order_known',
                        default=None, action='store_false')
    parser.add_argument('--redo', default=False, action='store_true')
    parser.add_argument('--verbosity', dest='verbose', default=1, type=int)
    parser.add_argument('--verbose', default=True, action='store_true')
    parser.add_argument('--silent', dest='verbose', default=True,
                        action='store_false')

    args = parser.parse_args()

    get_sc_disterror_all(
        struct_infer_files=args.struct_infer, sc_dis=args.dis_true,
        sc_features=args.features_true, rescale=args.rescale,
        hmlg_order_known=args.hmlg_order_known, redo=args.redo,
        verbose=args.verbose)


# if __name__ == "__main__":
#     main()


# Test

In [10]:
struct_infer_files = sorted(glob.glob(struct_infer_glob))
struct_infer_files = [x for x in struct_infer_files if 'alpha_infer-intrachr' not in x and 'multires' not in x]
struct_infer_files = [x for x in struct_infer_files if 'alpha_infer' not in x]
# struct_infer_files = [x for x in struct_infer_files if '.intramol' in x]
print('\n'.join(np.unique([os.path.basename(os.path.dirname(x)) for x in struct_infer_files])))
print('\n' + re.sub(f'.*{counts_subdir}/', '', struct_infer_files[0]) + '\n\n')

get_sc_disterror_all(
    struct_infer_files[0], sc_dis=sc_dis
    , sc_features=sc_features, rescale=True, hmlg_order_known=None,
    redo=False, verbose=True)

infer_ua_filter0perc.singleres

infer_ua_filter0perc.singleres/struct_inferred.000.coords


liu2025/unambig/astro.nreads1e9.logistic_k-5.16_m2.69_v1.23e-05/infer_ua_filter0perc.singleres/disterror_per_bin.true_vs_infer000.rescaled
Assuming homolog ordering of inferred structure has been pre-established
Comparing all distance bins
count    1654.000000
mean        1.310551
std         1.209011
min         0.318376
25%         0.831541
50%         0.900152
75%         1.055486
max         8.087185

5.0635028958603865 5.397038099383213 5.69083786772366 6.048499803246217
              i.idx_chrom  j.idx_chrom i.chrom  i.hmlg  dis_infer
idx                                                              
(1, 2)                  1            2    chr1       1   7.035282
(92, 93)               15           16    chr2       1   6.356920
(96, 97)               19           20    chr2       1   6.746016
(97, 98)               20           21    chr2       1   7.740371
(129, 130)             52      

ValueError: stop here

In [11]:
struct_infer_files = sorted(glob.glob(struct_infer_glob))
struct_infer_files = [x for x in struct_infer_files if 'alpha_infer-intrachr' not in x and 'multires' not in x]
struct_infer_files = [x for x in struct_infer_files if 'alpha_infer-intramol' in x]
# struct_infer_files = [x for x in struct_infer_files if '.intramol' in x]
print('\n'.join(np.unique([os.path.basename(os.path.dirname(x)) for x in struct_infer_files])))
print('\n' + re.sub(f'.*{counts_subdir}/', '', struct_infer_files[0]) + '\n\n')

get_sc_disterror_all(
    struct_infer_files[0], sc_dis=sc_dis
    , sc_features=sc_features, rescale=True, hmlg_order_known=None,
    redo=False, verbose=True)

infer_ua_filter0perc.alpha_infer-intramol.singleres

infer_ua_filter0perc.alpha_infer-intramol.singleres/struct_inferred.000.coords


liu2025/unambig/astro.nreads1e9.logistic_k-5.16_m2.69_v1.23e-05/infer_ua_filter0perc.alpha_infer-intramol.singleres/disterror_per_bin.true_vs_infer000.rescaled
Assuming homolog ordering of inferred structure has been pre-established
Comparing all distance bins
count    1654.000000
mean        2.671352
std         5.543989
min         0.622168
25%         0.937562
50%         1.076014
75%         1.351472
max        34.138455

17.46459394256123 21.412528511300643 23.059730478457766 26.010473086503545
              i.idx_chrom  j.idx_chrom i.chrom  i.hmlg  dis_infer
idx                                                              
(964, 965)              1            2    chr1       2  27.810399
(965, 966)              2            3    chr1       2  26.438972
(980, 981)             17           18    chr1       2  26.759821
(981, 982)             18      

ValueError: stop here

# Run

In [8]:
struct_infer_files = sorted(glob.glob(struct_infer_glob))

struct_infer_files = [x for x in struct_infer_files if 'alpha_infer-intrachr' not in x and 'multires' not in x]
print('\n'.join(np.unique([os.path.basename(os.path.dirname(x)) for x in struct_infer_files])))

get_sc_disterror_all(
    struct_infer_files, sc_dis=sc_dis
    , sc_features=sc_features, rescale=False, hmlg_order_known=None,
    redo=False, verbose=True)

infer_ua_filter0perc.alpha_infer-intramol.singleres
infer_ua_filter0perc.singleres
liu2025/unambig/astro.nreads1e9.logistic_k-9.12_m1.72_v1e-05/infer_ua_filter0perc.alpha_infer-intramol.singleres/disterror_per_bin.true_vs_infer000

liu2025/unambig/astro.nreads1e9.logistic_k-9.12_m1.72_v1e-05/infer_ua_filter0perc.alpha_infer-intramol.singleres/disterror_per_bin.true_vs_infer001
Assuming homolog ordering of inferred structure has been pre-established
Comparing all distance bins

liu2025/unambig/astro.nreads1e9.logistic_k-9.12_m1.72_v1e-05/infer_ua_filter0perc.alpha_infer-intramol.singleres/disterror_per_bin.true_vs_infer002
Assuming homolog ordering of inferred structure has been pre-established
Comparing all distance bins

liu2025/unambig/astro.nreads1e9.logistic_k-9.12_m1.72_v1e-05/infer_ua_filter0perc.alpha_infer-intramol.singleres/disterror_per_bin.true_vs_infer003
Assuming homolog ordering of inferred structure has been pre-established
Comparing all distance bins

liu2025/unambig/as

In [ ]:
%%bash

grep -E $'(alpha)\t' ~/noble_lab/projects/2015_gesine_diploid/results/liu2025/unambig/$counts_subdir/infer_ua_filter0perc.alpha_infer*/inference_params.* | sed 's/.*filter0perc.//;s/inference_params\.//'

# Temp

In [21]:
# get_sc_disterror_all(
#     struct_infer_file, sc_dis=sc_dis, hmlg_order_known=False,
#     sc_dis_are_scaled=True, redo=True, verbose=True)

/net/gs/vol1/home/gesine/noble_lab/projects/2015_gesine_diploid/results/liu2025/test_astro_1e6_750nm/infer_ua_filter0perc.singleres/disterror_per_bin.true_vs_infer000
Assuming homolog ordering of inferred structure has been pre-established
Comparing all distance bins
            disterror.vs_all disterror.vs_mean disterror.vs_med
all                 22.82978         20.004178        20.907642
diffM               4.841569          4.534009         4.635017
sameC-sameH         1.143295          0.723098         0.792076
sameC-diffH         5.298075          5.013404         5.096436
diffC-sameH         4.697406          4.380173         4.480505
diffC-diffH         4.953771          4.653551         4.756468


In [ ]:
import pandas as pd
import numpy as np
import ast

sc_dis_file = '/net/gs/vol1/home/gesine/noble_lab/repos/Chromatin_Analysis_MOp/cluster.LINK/astro/compare_to_1cell/matrix2d/astro.distances.per_locus.csv'

sc_dis = pd.read_csv(
    sc_dis_file, sep='\t', header=0, converters={
        0: ast.literal_eval, 1: str, 2: float, 3: float, 4: ast.literal_eval},
    names=('idx', 'same_molecule', 'dis_mean', 'dis_med', 'dis'))

sc_dis

sc_dis['same_molecule'] = sc_dis['same_molecule'].astype(int)

sc_dis

sc_dis.to_csv(sc_dis_file, index=False, header=False, sep='\t')

# sed 's/True/1/;s/False/0/' astro.distances.per_locus.csv > astro.distances.per_locus.csv.new